# Exercise 5: Model Versioning & Packaging

In this exercise, we will:
1. Register our tuned model in MLflow Model Registry
2. Merge LoRA weights with the base model for efficient serving
3. Create a Docker container for serving the model with vLLM
4. Build and test the container locally

---

## Prerequisites

Before starting this exercise, make sure you have completed:
- Exercise 1: Setup & Exploration
- Exercise 2: Data Preparation
- Exercise 3: LoRA Tuning
- Exercise 4: Evaluation

You should have a trained LoRA adapter saved and tracked in MLflow.

## 1. Environment Setup

Let's first check our environment and import necessary libraries.

In [ ]:
# Import necessary libraries
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import mlflow
import mlflow.pytorch
from pathlib import Path
from getpass import getpass

# Get Hugging Face token
hf_token = os.getenv("HF_TOKEN", None)
if hf_token:
    print("HF Token found")
else:
    print("Warning: No HF Token provided. You may encounter rate limits.")

# Check if we have GPU available
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current device: {torch.cuda.current_device()}")
else:
    print("Current device: CPU")


## 2. Load the Trained LoRA Model from MLflow

In this section, we will load our trained LoRA adapter from MLflow where we saved it during the training exercise.

In [ ]:
# Set up MLflow tracking
# In a real workshop, this would point to your MLflow tracking server
mlflow.set_tracking_uri("file://./mlruns")

# Experiment name used during training
experiment_name = "llm-lora-tuning"

# Get the experiment
experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    print(f"Experiment '{experiment_name}' not found. Please run the LoRA tuning exercise first.")
else:
    print(f"Experiment ID: {experiment.experiment_id}")

    # Search for runs in the experiment
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    print(f"Found {len(runs)} runs")

    if len(runs) > 0:
        # Get the latest run
        latest_run = runs.iloc[0]
        run_id = latest_run.run_id
        print(f"Latest run ID: {run_id}")
        print(f"Run status: {latest_run.status}")

        # Load the model from MLflow
        model_uri = f"runs:/{run_id}/model"
        print(f"Loading model from: {model_uri}")

        try:
            # Load the base model
            base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    # Use quantization for memory efficiency
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        token=hf_token,
    )
                base_model_name,
                torch_dtype=torch.float16,
                device_map="auto"
            )

            # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_name,
        token=hf_token,
    )

            # Load the LoRA model
            model = PeftModel.from_pretrained(base_model, model_uri)

            print("Model loaded successfully!")
            print(f"Model type: {type(model)}")
        except Exception as e:
            print(f"Error loading model: {e}")
            print("Make sure you have completed the LoRA tuning exercise and saved the model to MLflow.")

## 3. Register Model in MLflow Model Registry

Now we'll register our trained model in the MLflow Model Registry for versioning and lifecycle management.

In [ ]:
# Register the model in MLflow Model Registry
model_name = "tinyllama-lora-bike-assistant"

try:
    # Create the registered model if it doesn't exist
    client = mlflow.tracking.MlflowClient()
    try:
        client.create_registered_model(model_name)
        print(f"Created new registered model: {model_name}")
    except Exception:
        print(f"Registered model {model_name} already exists")

    # Create a new model version
    model_version = client.create_model_version(
        name=model_name,
        source=model_uri,
        run_id=run_id
    )
    print(f"Model version {model_version.version} created for {model_name}")

    # Optionally transition the model to a stage (e.g., Staging)
    client.transition_model_version_stage(
        name=model_name,
        version=model_version.version,
        stage="Staging",
        archive_existing_versions=False
    )
    print(f"Model version {model_version.version} transitioned to Staging")

except Exception as e:
    print(f"Error registering model: {e}")

## 4. Merge LoRA Weights with Base Model

For serving efficiency, we merge the LoRA adapter weights with the base model. This creates a single model file that doesn't require the PEFT library at inference time.

In [ ]:
# Merge LoRA weights with base model
print("Merging LoRA weights with base model...")

# Merge the LoRA layers into the base model
merged_model = model.merge_and_unload()

# Save the merged model
merged_model_path = "./merged_model"
os.makedirs(merged_model_path, exist_ok=True)

merged_model.save_pretrained(merged_model_path)
tokenizer.save_pretrained(merged_model_path)

print(f"Merged model saved to: {merged_model_path}")

# Verify the merged model loads correctly
print("Verifying merged model...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    test_model = AutoModelForCausalLM.from_pretrained(
        merged_model_path,
        quantization_config=bnb_config,
        device_map="auto",
        token=hf_token,
    )
    merged_model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
    test_tokenizer = AutoTokenizer.from_pretrained(
        merged_model_path,
        token=hf_token,
    )
print("Merged model verified successfully!")

## 5. Create Build and Push Script

Now we'll create a helper script for building and pushing the Docker image to a container registry.

In [ ]:
build_and_push_content = """#!/bin/bash
# Build and push Docker image for LLM serving

set -euo pipefail

# Default variables - override via environment or CLI
IMAGE_NAME="${IMAGE_NAME:-llm-instruction-tuning}"
VERSION="${VERSION:-v1.0.0}"
REGISTRY="${REGISTRY:-quay.io}"
NAMESPACE="${NAMESPACE:-your-namespace}"

usage() {
    echo "Usage: $0 [--name IMAGE_NAME] [--version VERSION] [--registry REGISTRY] [--namespace NAMESPACE]"
    echo ""
    echo "Build and push the LLM serving Docker image."
    echo "  --name       Image name (default: llm-instruction-tuning)"
    echo "  --version    Image tag (default: v1.0.0)"
    echo "  --registry   Container registry (default: quay.io)"
    echo "  --namespace  Registry namespace (default: your-namespace)"
    exit 1
}

# Parse arguments
while [[ $# -gt 0 ]]; do
    case $1 in
        --name) IMAGE_NAME="$2"; shift 2 ;;
        --version) VERSION="$2"; shift 2 ;;
        --registry) REGISTRY="$2"; shift 2 ;;
        --namespace) NAMESPACE="$2"; shift 2 ;;
        *) usage ;;
    esac
done

# Full image tag
FULL_TAG="${REGISTRY}/${NAMESPACE}/${IMAGE_NAME}:${VERSION}"

echo "Building image: ${FULL_TAG}"

# Build the image
docker build -t "${FULL_TAG}" .

echo "Image built successfully: ${FULL_TAG}"
echo ""
echo "To push to registry:"
echo "  docker push ${FULL_TAG}"
echo ""
echo "To run locally:"
echo "  docker run --gpus all -p 8000:8000 ${FULL_TAG}"
"""

with open('../scripts/build_and_push.sh', 'w') as f:
    f.write(build_and_push_content)

print("build_and_push.sh created in scripts/ directory")

## 6. Create MLflow Registration Script

Now we'll create a reusable script for registering models in MLflow.

In [ ]:
# Import necessary libraries
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import mlflow
import mlflow.pytorch
from pathlib import Path
from getpass import getpass

# Get Hugging Face token
hf_token = os.getenv("HF_TOKEN", None)
if hf_token:
    print("HF Token found")
else:
    print("Warning: No HF Token provided. You may encounter rate limits.")

# Check if we have GPU available
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current device: {torch.cuda.current_device()}")
else:
    print("Current device: CPU")


## 7. Verify the Scripts

Let's verify that the scripts were created correctly.

In [ ]:
import os

# Check if scripts exist
scripts_dir = os.path.join('..', 'scripts')
if os.path.exists(scripts_dir):
    files = os.listdir(scripts_dir)
    print(f"Scripts in {scripts_dir}:")
    for f in files:
        path = os.path.join(scripts_dir, f)
        size = os.path.getsize(path)
        print(f"  - {f} ({size} bytes)")
else:
    print(f"Directory {scripts_dir} does not exist yet.")

## 8. Clean Up Temporary Files

Let's clean up the temporary tokenizer directory we created.

In [ ]:
import shutil

# Remove temporary tokenizer directory
if os.path.exists('./tokenizer'):
    shutil.rmtree('./tokenizer')
    print("Cleaned up temporary tokenizer directory")

---

## Summary

In this exercise, we learned how to:
1. Register a fine-tuned model in MLflow Model Registry
2. Merge LoRA adapter weights with the base model for efficient serving
3. Create a build and push script for containerization
4. Create a reusable MLflow registration script

## Key Takeaways

- **MLflow Model Registry** provides version control and lifecycle management for ML models
- **LoRA weight merging** eliminates the PEFT dependency at inference time, reducing complexity
- **Containerization** packages the model with all dependencies for reproducible deployment
- **Script automation** ensures consistent and repeatable model registration and packaging

---

**Next: Proceed to Exercise 6 - Deployment & Serving**